In [5]:
import sqlite3
import random
import requests
import sys

API_KEY = "XK01BFB-ZMSMN4R-PJV5WVF-T1DWT89"
DB_NAME = "cinema.db"

def create_tables(conn):
    cursor = conn.cursor()
    cursor.execute("""
        CREATE TABLE IF NOT EXISTS movies (
            id INTEGER PRIMARY KEY,
            title TEXT NOT NULL,
            year INTEGER,
            genres TEXT,
            description TEXT,
            poster_url TEXT,
            rating_kp REAL,
            votes_kp INTEGER
        )
    """)
    cursor.execute("""
        CREATE TABLE IF NOT EXISTS ratings (
            id INTEGER PRIMARY KEY AUTOINCREMENT,
            user_id INTEGER NOT NULL,
            movie_id INTEGER NOT NULL,
            rating REAL NOT NULL,
            timestamp DATETIME DEFAULT CURRENT_TIMESTAMP,
            FOREIGN KEY(movie_id) REFERENCES movies(id)
        )
    """)
    conn.commit()

def fetch_movies_from_api(api_key: str, total_count: int = 500):
    url = "https://api.poiskkino.dev/v1.4/movie"
    headers = {"X-API-KEY": api_key}
    params = {
        "page": 1,
        "limit": total_count,
        "sortField": "votes.kp",
        "sortType": "-1",
        "type": "movie"
    }
    
    response = requests.get(url, headers=headers, params=params, timeout=10)
    
    if response.status_code != 200:
        raise RuntimeError(f"Ошибка API (код {response.status_code}): {response.text}")
    
    data = response.json()
    docs = data.get("docs", [])
    
    if not docs:
        raise ValueError("API вернул пустой список фильмов. Проверьте параметры запроса.")
    
    movies = []
    for item in docs:
        m_id = item.get("id")
        title = item.get("name") or item.get("alternativeName")
        if not title:
            continue
            
        year = item.get("year")
        genres = ", ".join([g.get("name") for g in item.get("genres", []) if g.get("name")])
        desc = item.get("description") or item.get("shortDescription") or ""
        poster = (item.get("poster") or {}).get("url") or ""
        rating_kp = (item.get("rating") or {}).get("kp") or 0.0
        votes_kp = (item.get("votes") or {}).get("kp") or 0
        
        movies.append((m_id, title, year, genres, desc, poster, rating_kp, votes_kp))
        
    return movies

def seed_ratings(conn, num_users=80):
    cursor = conn.cursor()
    cursor.execute("SELECT id FROM movies")
    movie_ids = [row[0] for row in cursor.fetchall()]
    
    if not movie_ids:
        return

    ratings_data = []
    for user_id in range(1, num_users + 1):
        num_rated = random.randint(10, min(35, len(movie_ids)))
        selected_movies = random.sample(movie_ids, num_rated)
        
        user_preference = random.choice([0, 1])
        for m_id in selected_movies:
            if m_id % 2 == user_preference:
                score = random.randint(7, 10)
            else:
                score = random.randint(2, 7)
            ratings_data.append((user_id, m_id, float(score)))

    cursor.executemany("""
        INSERT INTO ratings (user_id, movie_id, rating)
        VALUES (?, ?, ?)
    """, ratings_data)
    conn.commit()

def main():
    conn = sqlite3.connect(DB_NAME)
    create_tables(conn)
    
    print("Запрос данных к poiskkino.dev...")
    try:
        movies = fetch_movies_from_api(API_KEY, total_count=250)
    except Exception as e:
        print(f"Не удалось получить данные: {e}")
        conn.close()
        sys.exit(1)

    cursor = conn.cursor()
    cursor.executemany("""
        INSERT OR REPLACE INTO movies (id, title, year, genres, description, poster_url, rating_kp, votes_kp)
        VALUES (?, ?, ?, ?, ?, ?, ?, ?)
    """, movies)
    conn.commit()
    print(f"Загружено фильмов из API: {len(movies)}")

    seed_ratings(conn)
    print("Матрица оценок пользователей сгенерирована.")
    
    conn.close()
    print(f"База данных {DB_NAME} успешно создана.")

if __name__ == "__main__":
    main()

Запрос данных к poiskkino.dev...
Загружено фильмов из API: 250
Матрица оценок пользователей сгенерирована.
База данных cinema.db успешно создана.
